In [1]:
"""
Во время финансовых кризисов (2008, 2020) инвесторы сталкиваются с феноменом Contagion (эффект заражения). 
В обычное время акции IT-сектора и акции нефтяных компаний могут двигаться независимо. 
Но когда наступает шок (например, крах Lehman Brothers), корреляция всех активов стремится к единице (все падает одновременно).

Почему так происходит? Из-за механизма "Fire Sales" (Вынужденные распродажи).
Представьте крупный хедж-фонд, который терпит гигантские убытки из-за обвала технологического сектора. 
Брокер требует от фонда внести дополнительное обеспечение (Margin Call). Где фонду взять наличные? 
Он вынужден экстренно продавать "хорошие" активы (нефтянку, золото, гособлигации). 
Из-за этих распродаж падают цены активов, которые изначально вообще не были связаны с кризисом. 
Шок передается по рынку, как вирус.

Наша задача на семинаре:

Топология рынка: Мы возьмем 30 крупнейших компаний США из разных секторов и построим матрицу их корреляций во 
время Ковидного шока (весна 2020).
Фильтрация шума (Graph Theory): Матрица 30x30 — это 900 связей. Анализировать её невозможно. 
Мы применим теорию графов: превратим корреляцию в математическое расстояние и построим 
MST (Minimal Spanning Tree / Минимальное остовное дерево). 
Это выявит "хребет" рынка — узловые компании, через которые передается системный риск.
Стресс-тест: Мы напишем симулятор. Что будет с портфелем, 
если акция Apple завтра рухнет на 15%? 
Мы рассчитаем волну падения по всем остальным акциям, используя статистический аппарат условной регрессии.
Источник данных:
API yfinance (Yahoo Finance). Это бесплатная библиотека, которая напрямую скачивает биржевые данные.
Мы будем использовать Adjusted Close (Скорректированная цена закрытия), 
так как она учитывает выплату дивидендов и сплиты акций (что критически важно для корректного расчета доходности).
"""

_IncompleteInputError: incomplete input (969763094.py, line 1)

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Топ-30 компаний из разных секторов (Tech, Finance, Healthcare, Energy, Consumer)
tickers =[
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',          # Tech
    'JPM', 'BAC', 'WFC', 'C', 'GS',                   # Finance
    'JNJ', 'PFE', 'UNH', 'ABBV', 'MRK',               # Healthcare
    'XOM', 'CVX', 'COP', 'SLB', 'EOG',                # Energy
    'WMT', 'PG', 'KO', 'PEP', 'MCD',                  # Consumer
    'BA', 'CAT', 'MMM', 'GE', 'HON'                   # Industrials
]

print("Скачиваем данные с Yahoo Finance...")
# Скачиваем данные за период Ковидного шока (январь - июнь 2020)
data = yf.download(tickers, start="2020-01-01", end="2020-06-30")['Adj Close']
# Если yfinance вернул мультииндекс, убираем лишнее, оставляем только тикеры
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# TODO 1.1: Рассчитайте ежедневные логарифмические доходности (Log Returns).
# Подсказка: Формула лог-доходности: r_t = ln(P_t / P_{t-1}).
# Используйте np.log() и метод .shift(1) или метод .diff().
# Удалите первую строку (там будет NaN).
returns = # <ВАШ_КОД>

# TODO 1.2: Рассчитайте матрицу корреляций Пирсона для доходностей.
# Используйте встроенный метод pandas.
corr_matrix = # <ВАШ_КОД>

# Визуализация тепловой карты корреляций (готовый код)
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Матрица корреляций активов (Шок 2020 года)')
plt.show()


In [ ]:
import networkx as nx

# TODO 2.1: Трансформируйте матрицу корреляций в матрицу дистанций.
# Классическая финансовая формула: D = sqrt(2 * (1 - Correlation))
# Используйте np.sqrt(). Обратите внимание, что мы работаем с матрицей (DataFrame).
distance_matrix = # <ВАШ_КОД>

# Создаем пустой ненаправленный граф
G = nx.Graph()

# TODO 2.2: Заполните граф вершинами и ребрами.
# Напишите двойной цикл (по строкам и колонкам distance_matrix).
# Добавляйте ребро между тикерами (G.add_edge), только если это разные тикеры (i != j).
# В качестве атрибута 'weight' передавайте значение дистанции из матрицы.
for i in distance_matrix.columns:
    for j in distance_matrix.columns:
        if # <ВАШ_КОД>:
            G.add_edge(# <ВАШ_КОД: i, j, weight=...>)

# TODO 2.3: Постройте Минимальное остовное дерево (MST).
# Используйте функцию nx.minimum_spanning_tree() от вашего графа G.
mst = # <ВАШ_КОД>

# Визуализация MST (Код готов, так как настройка графов занимает много времени)
plt.figure(figsize=(12, 10))
# Располагаем узлы так, чтобы тесно связанные были рядом (Kamada Kawai layout)
pos = nx.kamada_kawai_layout(mst)

# Рисуем узлы и ребра
nx.draw_networkx_nodes(mst, pos, node_size=700, node_color='lightblue')
nx.draw_networkx_edges(mst, pos, width=2, alpha=0.6)
nx.draw_networkx_labels(mst, pos, font_size=10, font_family="sans-serif", font_weight='bold')

plt.title('Минимальное остовное дерево (MST) рынка', fontsize=15)
plt.axis('off')
plt.show()

# Дискуссия с группой:
# Посмотрите на граф. Видите ли вы секторальные кластеры? 
# Кто находится в самом центре графа (Hub)? Именно через эти акции шок распространяется на остальные сектора.


In [ ]:
# TODO 3.1: Рассчитайте вектор стандартных отклонений (волатильностей) для всех активов.
# Используйте метод .std() к датафрейму returns.
volatilities = # <ВАШ_КОД>

# Задаем параметры шока
shocked_asset = 'AAPL'  # Акция, с которой начался кризис
shock_value = -0.10     # Падение на 10%

# Создаем пустой словарь для записи того, как упадут остальные активы
contagion_effects = {}

# TODO 3.2: Напишите цикл для расчета ожидаемого падения каждого актива.
# Пройдитесь циклом по всем тикерам (target_asset) в списке tickers.
# Если target_asset == shocked_asset, его падение равно shock_value.
# Для остальных:
# 1. Извлеките корреляцию между shocked_asset и target_asset из corr_matrix.
# 2. Вычислите бету по математической формуле: Beta = Correlation * (Sigma_target / Sigma_shocked).
# 3. Рассчитайте ожидаемую доходность: Expected_Return = Beta * shock_value.
for target_asset in tickers:
    if target_asset == shocked_asset:
        contagion_effects[target_asset] = shock_value
    else:
        # <ВАШ_КОД: Извлечение корреляции>
        correlation = # <ВАШ_КОД>
        
        # <ВАШ_КОД: Расчет беты>
        beta = # <ВАШ_КОД>
        
        # <ВАШ_КОД: Расчет ожидаемого падения>
        expected_return = # <ВАШ_КОД>
        
        contagion_effects[target_asset] = expected_return

# Превращаем словарь в DataFrame и сортируем для красоты
shock_df = pd.DataFrame.from_dict(contagion_effects, orient='index', columns=['Expected_Return'])
shock_df = shock_df.sort_values(by='Expected_Return')

# Визуализация результатов стресс-теста
plt.figure(figsize=(12, 6))
# Выделим источник шока красным цветом
colors =['red' if x == shocked_asset else 'steelblue' for x in shock_df.index]
sns.barplot(x=shock_df.index, y=shock_df['Expected_Return'] * 100, palette=colors)
plt.title(f'Стресс-тест: Эффект заражения при падении {shocked_asset} на {shock_value*100}%')
plt.ylabel('Ожидаемое изменение цены (%)')
plt.xticks(rotation=90)
plt.grid(axis='y', linestyle='--')
plt.show()
